## Bronze Layer — Raw Ingestion

In [0]:
from pyspark.sql import functions as F

storage_account = "healthflowstorage"
container = "raw"

# Set storage config
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    dbutils.secrets.get(
        scope="healthflow",
        key="storage-key"
    )
)

# Read CSV
df = spark.read.csv(
    f"wasbs://{container}@{storage_account}.blob.core.windows.net/healthcare_dataset.csv",
    header=True,
    inferSchema=True
)

print("Original columns:", df.columns)

# Rename ALL columns
bronze_df = df \
    .withColumnRenamed("Blood Type", "blood_type") \
    .withColumnRenamed("Medical Condition", "medical_condition") \
    .withColumnRenamed("Date of Admission", "date_of_admission") \
    .withColumnRenamed("Insurance Provider", "insurance_provider") \
    .withColumnRenamed("Billing Amount", "billing_amount") \
    .withColumnRenamed("Room Number", "room_number") \
    .withColumnRenamed("Admission Type", "admission_type") \
    .withColumnRenamed("Discharge Date", "discharge_date") \
    .withColumnRenamed("Test Results", "test_results") \
    .withColumn("ingestion_timestamp", F.current_timestamp()) \
    .withColumn("source_file", F.lit("healthcare_dataset.csv"))

print("Clean columns:", bronze_df.columns)
print(f"Total records: {bronze_df.count():,}")

# Save to bronze
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "healthflow_catalog.bronze.patients_raw"
    )

print("✅ Bronze layer complete!")

Original columns: ['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition', 'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date', 'Medication', 'Test Results']
Clean columns: ['Name', 'Age', 'Gender', 'blood_type', 'medical_condition', 'date_of_admission', 'Doctor', 'Hospital', 'insurance_provider', 'billing_amount', 'room_number', 'admission_type', 'discharge_date', 'Medication', 'test_results', 'ingestion_timestamp', 'source_file']
Total records: 55,500
✅ Bronze layer complete!
